## DFT vs. MLIPs Energy Comparison Pipeline

In [37]:
from pathlib import Path
import numpy as np
from ase.io import iread, read, write

#### 0. Configuration

In [38]:
def make_run(label, bext, ads=None, slab=None, gas=None):
    def path(snippet, default):
        if snippet and '/' in snippet:
            return snippet  # fully specified path, use as-is
        name = snippet or label + default
        return f"dft-data/{name}_BEXT{bext}/vasprun.xml"
    return {
        "label": label,
        "bext":  bext,
        "ads":   path(ads,  "diss2"),
        "slab":  path(slab, "surf"),
        "gas":   path(gas,  "mag2"),
    }

In [39]:
# USER INPUT HERE

RUNS = ([make_run("NH3", b) for b in [0.1, 0.24, 0.37, 0.5]] +
        [make_run("H2", b, slab="NH3surf", gas="dft-data/H2_gas/vasprun.xml") for b in [0.1, 0.24, 0.37, 0.5]]
)

MODELS = [
        ("MACE",       "/Users/zschwab/miniconda3/envs/mlip-mace/bin/python",       "model-scripts/run_mace.py"),
        ("MatterSim",  "/Users/zschwab/miniconda3/envs/mlip-mattersim/bin/python",  "model-scripts/run_mattersim.py"),
        ("UMA",        "/Users/zschwab/miniconda3/envs/mlip-uma/bin/python",        "model-scripts/run_uma.py"),
]

#### 1. DFT data ingestion

In [40]:
# load vasp trajectories into ASE Atoms obj.
# metadata: (system type, source dataset, external b-field strength)

def load_frames(path):
    frames = []
    for f in iread(path):
        frames.append(f)
    
    return frames

def load_frame(path):
    frame = read(path)
    
    return frame

In [41]:
for run in RUNS:
    run_id = f"{run['label']}_BEXT{run['bext']}"
    
    run["xyz_ads"]  = f"model-scripts/script-data/input_ads_{run_id}.xyz"
    run["xyz_slab"] = f"model-scripts/script-data/input_slab_{run_id}.xyz"
    run["xyz_gas"]  = f"model-scripts/script-data/input_gas_{run_id}.xyz"

    write(run["xyz_ads"],  load_frames(run["ads"]))
    write(run["xyz_slab"], load_frame (run["slab"]))
    write(run["xyz_gas"],  load_frame (run["gas"]))

#### 2. run MLIPs (MACE, MatterSim, UMA)

In [ ]:
import subprocess
import os
env = os.environ.copy()
env["MPLBACKEND"] = "Agg"

for run in RUNS:
    run_id = f"{run['label']}_BEXT{run['bext']}"
    run["results"] = {}
    for name, python, script in MODELS:
        out = f"model-scripts/script-data/results_{name}_{run_id}.npz"
        r = subprocess.run(
            [python, script, run["xyz_ads"], run["xyz_slab"], run["xyz_gas"], out],
            capture_output=True, text=True, env=env
        )
        run["results"][name] = out
        if (r.returncode != 0):
            print(f"-- [{run_id}] ({name}): failed")
            if r.stdout: print(r.stdout)
            if r.stderr: print(r.stderr)
            print("Return code:", r.returncode)
        print(f"-- [{run_id}] ({name}): success")

-- [NH3_BEXT0.1] (MACE): success
-- [NH3_BEXT0.1] (MatterSim): success
-- [NH3_BEXT0.1] (UMA): success
